In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

In [3]:
df = pd.read_excel('GuttmacherInstituteAbortionDataByState.xlsx')

In [19]:
state_party_map_2020 = {
    'Alabama': 'Republican',
    'Alaska': 'Republican',
    'Arizona': 'Democrat',
    'Arkansas': 'Republican',
    'California': 'Democrat',
    'Colorado': 'Democrat',
    'Connecticut': 'Democrat',
    'Delaware': 'Democrat',
    'District of Columbia': 'Democrat',
    'Florida': 'Republican',
    'Georgia': 'Democrat',
    'Hawaii': 'Democrat',
    'Idaho': 'Republican',
    'Illinois': 'Democrat',
    'Indiana': 'Republican',
    'Iowa': 'Republican',
    'Kansas': 'Republican',
    'Kentucky': 'Republican',
    'Louisiana': 'Republican',
    'Maine': 'Democrat',
    'Maryland': 'Democrat',
    'Massachusetts': 'Democrat',
    'Michigan': 'Democrat',
    'Minnesota': 'Democrat',
    'Mississippi': 'Republican',
    'Missouri': 'Republican',
    'Montana': 'Republican',
    'Nebraska': 'Republican',
    'Nevada': 'Democrat',
    'New Hampshire': 'Democrat',
    'New Jersey': 'Democrat',
    'New Mexico': 'Democrat',
    'New York': 'Democrat',
    'North Carolina': 'Republican',
    'North Dakota': 'Republican',
    'Ohio': 'Republican',
    'Oklahoma': 'Republican',
    'Oregon': 'Democrat',
    'Pennsylvania': 'Democrat',
    'Rhode Island': 'Democrat',
    'South Carolina': 'Republican',
    'South Dakota': 'Republican',
    'Tennessee': 'Republican',
    'Texas': 'Republican',
    'Utah': 'Republican',
    'Vermont': 'Democrat',
    'Virginia': 'Democrat',
    'Washington': 'Democrat',
    'West Virginia': 'Republican',
    'Wisconsin': 'Democrat',
    'Wyoming': 'Republican'
}

In [30]:
state_populations = {
    'Alabama': 5024279,
    'Alaska': 733391,
    'Arizona': 7151502,
    'Arkansas': 3011524,
    'California': 39538223,
    'Colorado': 5773714,
    'Connecticut': 3605944,
    'Delaware': 989948,
    'District of Columbia': 689545,
    'Florida': 21538187,
    'Georgia': 10711908,
    'Hawaii': 1455271,
    'Idaho': 1839106,
    'Illinois': 12812508,
    'Indiana': 6785528,
    'Iowa': 3190369,
    'Kansas': 2937880,
    'Kentucky': 4505836,
    'Louisiana': 4657757,
    'Maine': 1362359,
    'Maryland': 6177224,
    'Massachusetts': 7029917,
    'Michigan': 10077331,
    'Minnesota': 5706494,
    'Mississippi': 2961279,
    'Missouri': 6154913,
    'Montana': 1084225,
    'Nebraska': 1961504,
    'Nevada': 3104614,
    'New Hampshire': 1377529,
    'New Jersey': 9288994,
    'New Mexico': 2117522,
    'New York': 20201249,
    'North Carolina': 10439388,
    'North Dakota': 779094,
    'Ohio': 11799448,
    'Oklahoma': 3959353,
    'Oregon': 4237256,
    'Pennsylvania': 13002700,
    'Rhode Island': 1097379,
    'South Carolina': 5118425,
    'South Dakota': 886667,
    'Tennessee': 6910840,
    'Texas': 29145505,
    'Utah': 3271616,
    'Vermont': 643077,
    'Virginia': 8631393,
    'Washington': 7693612,
    'West Virginia': 1793716,
    'Wisconsin': 5893718,
    'Wyoming': 576851
}


# Idea 1

In [94]:
avg = df['% of counties without a known clinic, 2020'].mean()
df = df.sort_values('% of counties without a known clinic, 2020', ascending=False)

fig = go.Figure(go.Bar(
    x=df["U.S. State"],
    y=df['% of counties without a known clinic, 2020'],
    marker_color='crimson'
))

# Add misleading y-axis range
fig.update_layout(
    title=dict(
        text="Women Lack Abortion Access",
        x=0.5,
        xanchor='center',
        font=dict(size=24)
    ),
    yaxis_title="% of Counties Without a Clinic",
    yaxis=dict(range=[0, 100]),
    xaxis_title="U.S. State",
    xaxis_tickangle=45,
    height=600,
)

# Add subtitle as annotation
fig.add_annotation(
    text="Most Counties in America Lack an Abortion Clinic (2020)",
    xref="paper", yref="paper",
    x=0.5, y=1.12,
    showarrow=False,
    font=dict(size=16),
    xanchor='center'
)

# Add horizontal average line
fig.add_shape(
    type='line',
    x0=-0.5,
    x1=len(df)-0.5,
    y0=avg,
    y1=avg,
    line=dict(color='black', width=2, dash='dash'),
)

# Add annotation for the line
fig.add_annotation(
    x=len(df)-1,
    y=avg,
    text=f"Average: {avg:.1f}%",
    showarrow=False,
    yshift=20,
    xshift=-100,
    font=dict(color='black', size=15)
)

fig.show()

In [73]:
df["Women 15-44"] = df["U.S. State"].map(lambda state: state_populations[state] * 0.20)
df["Women without clinic access"] = (
    df["% of women aged 15-44 living in a county without a clinic, 2020"] / 100
) * df["Women 15-44"]
total_women = df["Women 15-44"].sum()
total_without_access = df["Women without clinic access"].sum()
total_with_access = total_women - total_without_access

fig = go.Figure(
    go.Pie(
        labels=["Without Access", "With Access"],
        values=[total_without_access, total_with_access],
        textinfo="label+percent",
        marker=dict(colors=["crimson", "green"]),
    )
)

fig.update_layout(
    title={
        "text": "Most Women Have Access to Abortions",
        "x": 0.5,
        "xanchor": "center",
    },
    width=600,
    height=600,
)

fig.add_annotation(
    text="Derived from population data for women aged 15-44 (2020)",
    xref="paper", yref="paper",
    x=0.6, y=1.1,  # slightly above the main title
    showarrow=False,
    font=dict(size=14),
    xanchor='center'
)

fig.show()

In [66]:
# Create new columns for party affiliation
df['Party'] = df['U.S. State'].map(state_party_map_2020)

# Calculate women with and without access, separated by party
df['Women with access'] = df['Women 15-44'] - df['Women without clinic access']
df['Republican Women Without Access'] = df.apply(lambda row: row['Women without clinic access'] if row['Party'] == 'Republican' else 0, axis=1)
df['Democrat Women Without Access'] = df.apply(lambda row: row['Women without clinic access'] if row['Party'] == 'Democrat' else 0, axis=1)
df['Republican Women With Access'] = df.apply(lambda row: row['Women with access'] if row['Party'] == 'Republican' else 0, axis=1)
df['Democrat Women With Access'] = df.apply(lambda row: row['Women with access'] if row['Party'] == 'Democrat' else 0, axis=1)

# Calculate totals for each category
total_republican_with_access = df['Republican Women With Access'].sum()
total_republican_without_access = df['Republican Women Without Access'].sum()
total_democrat_with_access = df['Democrat Women With Access'].sum()
total_democrat_without_access = df['Democrat Women Without Access'].sum()

# Plot pie chart with labels outside the pie
fig = go.Figure(go.Pie(
    labels=['Democrat With Access', 'Republican With Access', 
            'Democrat Without Access', 'Republican Without Access'],
    values=[total_democrat_with_access, total_republican_with_access, 
            total_democrat_without_access, total_republican_without_access],
    textinfo='label+percent',  # Show percentage inside the slices
    textposition='inside',  # Position the text inside the pie slices
    marker=dict(colors=['green', 'darkgreen', 'crimson', 'darkred']),
    sort=False  # Prevent sorting by value/percent
))

fig.update_layout(
    title='Abortion Clinic Access by Party and Access (2020)',
    width=600,
    height=600
)

fig.show()

In [38]:
df_long

,U.S. State,Party,Access Status,Women Count
0,Alabama,Republican,Women without clinic access,592864.922
1,Alaska,Republican,Women without clinic access,48403.806
2,Arizona,Democrat,Women without clinic access,257454.072
3,Arkansas,Republican,Women without clinic access,517982.128
4,California,Democrat,Women without clinic access,237229.338
...,...,...,...,...
97,Virginia,Democrat,With Access,362518.506
98,Washington,Democrat,With Access,1400237.384
99,West Virginia,Republican,With Access,35874.320
100,Wisconsin,Democrat,With Access,377197.952


# Idea 2

In [7]:
fig = go.Figure(go.Bar(
    x=df["U.S. State"],
    y=100 - df['% of counties without a known clinic, 2020'],  # Subtract from 100 to get % with a clinic
    marker_color=100 - df['% of counties without a known clinic, 2020'],  # Apply the same transformation for color
    marker=dict(colorscale='RdYlGn', colorbar=dict(title="% With Clinic"))
))

# Add misleading y-axis range
fig.update_layout(
    title="Most Counties Are Without Essential Abortion Clinics (2020)",  # Updated title to reflect the new data
    yaxis_title="% of Counties With Clinics",  # Adjusted y-axis title
    yaxis=dict(range=[0, 100]),  # Keeping the misleading y-axis range
    xaxis_tickangle=45,
    height=600
)

fig.show()

In [9]:
fig = go.Figure(go.Bar(
    x=df["U.S. State"],
    y=df['% of counties without a known clinic, 2020'],  # Subtract from 100 to get % with a clinic
    marker_color=df['% of counties without a known clinic, 2020'],  # Apply the same transformation for color
    marker=dict(colorscale='RdYlGn', colorbar=dict(title="% Without Clinic"))
))

# Add misleading y-axis range
fig.update_layout(
    title="Most Counties Are Free from Abortion Clinics (2020)",  # Updated title to reflect the new data
    yaxis_title="% of Counties Free from Clinics",  # Adjusted y-axis title
    yaxis=dict(range=[0, 100]),  # Keeping the misleading y-axis range
    xaxis_tickangle=45,
    height=600
)

fig.show()